# PDF QA Bot (RAG) with LangChain + OpenAI

This notebook builds a retrieval-augmented QA bot for PDF documents using OpenAI for embeddings and the LLM.

Pipeline:
1. Load a PDF into LangChain `Document` objects.
2. Split documents into chunks.
3. Embed chunks with OpenAI embeddings.
4. Store embeddings in a Chroma vector database.
5. Retrieve relevant chunks for a query.
6. Answer with an OpenAI chat model using `RetrievalQA`.
7. Expose a simple Gradio UI.


## Setup

Reduce noisy warnings so notebook output focuses on the RAG pipeline.


In [1]:
import warnings
warnings.filterwarnings("ignore")

## Environment Variables

This notebook expects an OpenAI API key in a `.env` file (or your shell environment):
- `OPENAI_API_KEY`

If `python-dotenv` is installed, this notebook will auto-load `.env`.


In [2]:
import os
import sys
import importlib.util

print("Kernel Python:", sys.executable)

def _load_env_if_possible():
    if importlib.util.find_spec("dotenv") is not None:
        from dotenv import load_dotenv
        load_dotenv(override=True)

_load_env_if_possible()

openai_api_key = os.getenv("OPENAI_API_KEY")
if not openai_api_key:
    raise RuntimeError(
        "OPENAI_API_KEY is not set.\n"
        "Create a .env file with:\n"
        "  OPENAI_API_KEY=sk-...\n"
        "and ensure it is loaded before running the notebook."
    )

print("OPENAI_API_KEY loaded ✅ (prefix:", openai_api_key[:6], "…)")

Kernel Python: /Users/davidjbrady/venvs/ml_3.11.9_venv/bin/python3.11
OPENAI_API_KEY loaded ✅ (prefix: sk-pro …)


## Imports

Core components:
- `PyPDFLoader` for PDF loading
- `RecursiveCharacterTextSplitter` for chunking
- `OpenAIEmbeddings` for vectorization
- `Chroma` as the vector database
- `RetrievalQA` as the RAG QA chain
- `gradio` for the front-end UI


In [3]:
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_community.document_loaders import PyPDFLoader
from langchain_classic.chains import RetrievalQA

import gradio as gr

## Step 1: PDF Document Loader

Load a PDF into a list of LangChain `Document` objects.

The loader accepts either:
- A string file path (used by Gradio when `type="filepath"`)
- An object with a `.name` attribute


In [4]:
## Document loader
def document_loader(file):
    file_path = file if isinstance(file, str) else file.name
    loader = PyPDFLoader(file_path)
    loaded_document = loader.load()
    return loaded_document

## Step 2: Text Splitter

Split loaded documents into smaller overlapping chunks to improve retrieval quality.


In [5]:
## Text splitter
def text_splitter(data):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=100,
        chunk_overlap=20,
        length_function=len,
    )
    chunks = text_splitter.split_documents(data)
    return chunks

## Step 3: Embedding Model (OpenAI)

Convert chunks into vectors with an OpenAI embedding model.

Default:
- `text-embedding-3-small`


In [6]:
## Embedding model
def openai_embedding():
    return OpenAIEmbeddings(model="text-embedding-3-small")

## Step 4: Vector Store (Chroma)

Embed the chunks and store them in Chroma so we can run similarity search during retrieval.


In [7]:
## Vector db
def vector_database(chunks):
    embedding_model = openai_embedding()
    vectordb = Chroma.from_documents(chunks, embedding_model)
    return vectordb

## Step 5: Retriever

Create a vector-store-backed retriever that fetches relevant chunks via similarity search.


In [8]:
## Retriever
def retriever(file):
    splits = document_loader(file)
    chunks = text_splitter(splits)
    vectordb = vector_database(chunks)
    retriever = vectordb.as_retriever()
    return retriever

## Step 6a: LLM (OpenAI)

Build a chat LLM for answer generation.

Default:
- `gpt-4o-mini`


In [9]:
## LLM
def get_llm(model="gpt-4o-mini"):
    return ChatOpenAI(
        model=model,
        temperature=0.5,
        max_tokens=256,
    )

## Step 6b: QA Chain (RetrievalQA)

`RetrievalQA` combines retrieval + generation. We use `chain_type="stuff"` to provide retrieved chunks as context to the LLM.


In [10]:
## QA Chain
def retriever_qa(file, query):
    llm = get_llm()
    retriever_obj = retriever(file)
    qa = RetrievalQA.from_chain_type(
        llm=llm,
        chain_type="stuff",
        retriever=retriever_obj,
        return_source_documents=True,
    )
    try:
        response = qa.invoke(query)
    except Exception:
        response = qa.invoke({"query": query})
    return response["result"]

## Step 7: Gradio Interface

UI requirements:
- PDF upload
- Input textbox for the question
- Output textbox for the answer


In [11]:
# Create Gradio interface
rag_application = gr.Interface(
    fn=retriever_qa,
    #allow_flagging="never",
    inputs=[
        gr.File(label="Upload PDF File", file_count="single", file_types=['.pdf'], type="filepath"),
        gr.Textbox(label="Input Query", lines=2, placeholder="Type your question here...")
    ],
    outputs=gr.Textbox(label="Output"),
    title="PDF QA Bot (RAG)",
    description="Upload a PDF document and ask any question. The chatbot will try to answer using the provided document."
)

## Launch

- `debug=True` prints detailed exceptions in the notebook output.
- `show_error=True` (if supported by your Gradio version) also shows the traceback in the UI.


In [ ]:
# Launch the app
launch_kwargs = dict(server_name="localhost", server_port=6869, debug=True)
try:
    rag_application.launch(show_error=True, **launch_kwargs)
except TypeError:
    # Fallback for older Gradio versions that don't support `show_error`.
    rag_application.launch(**launch_kwargs)

* Running on local URL:  http://localhost:6869
* To create a public link, set `share=True` in `launch()`.
